# Exploratory Data Analysis — NYC Airbnb

This notebook loads the raw `sample.csv` artifact from Weights & Biases and
performs a lightweight EDA. Key decisions taken here are then transferred to the
`basic_cleaning` MLflow step:

1. Restrict nightly prices to the interval **\$10 – \$350** (stakeholder decision).
2. Convert `last_review` from string to `datetime`.
3. Filter rows to the geographical boundaries of NYC.

> Missing-value imputation is **not** performed here — that will happen in the
> inference pipeline so the model can handle missing values at prediction time.

In [ ]:
import wandb
import pandas as pd

run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)
df.head()

## Profile the raw dataset

We use **ydata-profiling** (the maintained successor of pandas-profiling).

In [ ]:
import ydata_profiling

profile = ydata_profiling.ProfileReport(df, title="Raw sample.csv profile")
profile.to_notebook_iframe()

### Findings

- Several columns contain **missing values** (`name`, `host_name`, `last_review`, `reviews_per_month`).
- `last_review` is stored as **string**, should be datetime.
- `price` has extreme outliers on both ends (zeros and thousands of dollars).

### Decisions

After discussing with stakeholders, we restrict prices to **\$10 – \$350**.

In [ ]:
# Drop price outliers
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()

# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

df.info()

### Verify

- ✅ All prices now inside `[10, 350]`.
- ✅ `last_review` is `datetime64[ns]`.

In [ ]:
assert df['price'].between(min_price, max_price).all()
assert df['last_review'].dtype.name.startswith('datetime')
print('All checks passed. Rows:', len(df))

In [ ]:
run.finish()